# SegFormer-B0 defect segmentation (Kaggle, T4 x2)

**Settings:** Accelerator = GPU T4 x2, Internet = ON, Persistence = Files only.
**Inputs to add:** the `defect-scripts` dataset (train_segformer.py + prepare_data.py) and any Kaggle crack datasets (e.g. via *Add Input* from the DeepCrack / Crack500 notebooks' data sources).

In [ ]:
!nvidia-smi
!ls /kaggle/input

In [ ]:
!pip install -q transformers albumentations

## CrackSeg9k (Harvard Dataverse, not on Kaggle)
Run once. Prints file sizes first. After it finishes, use **Save Version -> Output -> New Dataset** so later runs skip the download.

In [ ]:
import requests, os, zipfile
BASE, PID = "https://dataverse.harvard.edu", "doi:10.7910/DVN/EGIEBY"
RAW = "/kaggle/working/crackseg9k_raw"
os.makedirs(RAW, exist_ok=True)

meta = requests.get(f"{BASE}/api/datasets/:persistentId/", params={"persistentId": PID}, timeout=60).json()
files = meta["data"]["latestVersion"]["files"]
tot = 0
for f in files:
    d = f["dataFile"]; tot += d.get("filesize", 0)
    print(f'{d["id"]:>10}  {d.get("filesize", 0)/1e9:6.2f} GB  {d["filename"]}')
print(f"total {tot/1e9:.2f} GB  (/kaggle/working limit is ~20 GB; extracted size will be larger than the zips)")


In [ ]:
# only run after checking the sizes above
for f in files:
    d = f["dataFile"]; out = f"{RAW}/{d['filename']}"
    if not os.path.exists(out):
        with requests.get(f"{BASE}/api/access/datafile/{d['id']}", stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(out, "wb") as fh:
                for chunk in r.iter_content(1 << 20): fh.write(chunk)
    if out.endswith(".zip"):
        with zipfile.ZipFile(out) as z: z.extractall(RAW)
        os.remove(out)   # free space
!find {RAW} -maxdepth 3 -type d | head -30


## Config + data prep

In [ ]:
# Roots to scan. Auto-detect finds (images, masks) folder pairs inside each.
AUTO = {
    "crackseg9k": "/kaggle/working/crackseg9k_raw",
    # "deepcrack":  "/kaggle/input/<folder from ls above>",
    # "crack500":   "/kaggle/input/<folder from ls above>",
}
SCRIPTS = "/kaggle/input/defect-scripts"
DATA, RUN = "/kaggle/working/data", "/kaggle/working/runs/b0_v1"
EPOCHS, BS, LR = 50, 16, 6e-5   # BS is PER GPU (T4x2 -> effective 32)


In [ ]:
import shutil, subprocess
for f in ("train_segformer.py", "prepare_data.py"):
    shutil.copy(f"{SCRIPTS}/{f}", f"/kaggle/working/{f}")
os.chdir("/kaggle/working")
auto = sum([["--auto", f"{n}={p}"] for n, p in AUTO.items()], [])
subprocess.run(["python", "prepare_data.py", "--dry_run", *auto], check=True)   # CHECK this output: pairs must look right


In [ ]:
if not os.path.isdir(f"{DATA}/train/images"):
    subprocess.run(["python", "prepare_data.py", "--out", DATA, "--max_side", "1024", "--val_frac", "0.1", *auto], check=True)
else:
    print("data already prepared")


## Train (DDP on both T4s)

In [ ]:
# Resumes automatically if last.pt exists (after the 12h limit: attach previous output as input, copy last.pt into RUN)
# workers=2 per process because Kaggle only gives ~4 CPU cores.
resume = f"--resume {RUN}/last.pt" if os.path.exists(f"{RUN}/last.pt") else ""
!torchrun --nproc_per_node=2 train_segformer.py --data_root {DATA} --out_dir {RUN} --img_size 512 \
    --batch_size {BS} --epochs {EPOCHS} --lr {LR} --workers 2 {resume}
# If NCCL hangs/errors on init, rerun with:  !NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 torchrun ...


In [ ]:
import json
m = json.load(open(f"{RUN}/metrics.json"))
best = max(m, key=lambda x: x["mIoU"])
print("best epoch:", best["epoch"], {k: round(v, 4) for k, v in best.items() if k != "epoch"})
shutil.make_archive("/kaggle/working/segformer_b0_best", "zip", f"{RUN}/best")   # download from Output tab
